# Лабораторная 2. Статистика на отрезках

В первой лабораторной вы научились считать среднее, дисперсию, стандартное отклонение,
асимметрию и строить префиксы для `x`, `x²`, `x³`.

## Правила

- Нумерация элементов в запросах начинается с **1**.
- Границы `[l, r]` включаются в отрезок.
- Дисперсия считается с делением на количество элементов `n`.
- Префиксы нужно строить **один раз**.
- Каждый запрос должен обрабатываться за `O(1)`.
- Нельзя использовать `numpy`, `pandas`, `statistics`, `scipy`.
- Не меняйте имена функций и файлы в папке `tests`.

# Простые задачи на понимание метрик

Зачем в прошлом задании учились считать разные метрики? Для обучения искусственного интеллекта нужно подготовить данные. В них бывают разные выбросы, из-за которых можно переобучить модель. Из-за чего искусственный интеллект будет постоянно ошибаться в своих ответах. Чтобы такого не происходило нужно уметь анализировать данные (например, получать характер распределения, но об этом будет в следующих заданиях) и удалять лишние данные, например, выбросы, то есть те элементы, которые далеко находятся от "центра" данных. В первых трех задачах будем убирать выбросы и анализировать характер распределения данных. 

Здесь не нужно думать об эффективности на больших данных. Главная цель — понять,
**что именно означают среднее, стандартное отклонение и асимметрия**.


## Задача 0A. Удаляем далёкие от среднего значения

В школьной метеостанции каждый час записывается температура воздуха. Иногда датчик ошибается и выдаёт слишком большое или слишком маленькое значение.
Известно, что нормальные показания обычно находятся недалеко от среднего значения за весь период наблюдений.
Вам дан список измерений и число max_distance. Нужно удалить все показания, которые отличаются от среднего значения больше чем на max_distance.
Порядок оставшихся измерений менять нельзя.

### Пример

```text
values = [10, 11, 12, 13, 50]
max_distance = 10

Среднее = 19.2

Ответ:
[10, 11, 12, 13]
```

Число `50` слишком далеко от центра массива.

Для пустого массива верните `[]`.

In [5]:
def filter_by_mean_distance(values, max_distance):
    """Оставляет элементы, находящиеся не дальше max_distance от среднего."""
    # TODO
    if not values:
        return []
    s = 0
    for i in values:
        s += i
    av = s / len(values)
    a = []
    for i in values:
        if abs(av - i) <= max_distance:
            a.append(i)
    return a



In [6]:
%run -i ./tests/check_0a_mean_filter.py

OK: задача 0A


## Задача 0B. Удаляем выбросы с помощью стандартного отклонения

На автоматизированном заводе робот проверяет массу каждой детали. Из-за ошибок оборудования иногда встречаются детали, масса которых слишком сильно отличается от остальных.
Инженеры решили считать подозрительными те детали, масса которых отличается от среднего значения больше чем на k стандартных отклонений.
Для массива измерений values нужно оставить только те значения x, которые удалены от среднего значения не больше, чем на k стандартных отклонений.

### Пример

```text
values = [10, 10, 11, 9, 30]
k = 1

Ответ:
[10, 10, 11, 9]
```

Если все элементы одинаковые, стандартное отклонение равно нулю.
В таком случае нужно оставить все элементы.

Для пустого массива верните `[]`.

In [9]:
import math


def filter_by_std(values, k):
    """Оставляет элементы не дальше k стандартных отклонений от среднего."""
    # TODO
    if not values:
        return []
    s = 0
    s1 = 0
    for i in values:
        s += i
        s1 += i * i
    av = s / len(values)
    var = s1 / len(values) - av * av
    std = math.sqrt(var)
    a = []
    for i in values:
        if abs(av - i) <= std * k:
            a.append(i)
    return a

In [10]:
%run -i ./tests/check_0b_std_filter.py

OK: задача 0B


## Задача 0C. Куда направлен «хвост»?

В королевстве вдоль одной дороги построены башни разной высоты. Королевский архитектор хочет понять, в какую сторону «смещено» распределение высот.
Если среди башен встречаются несколько очень высоких, распределение имеет длинный хвост вправо. Если, наоборот, есть несколько очень низких башен, хвост направлен влево. Определите, в какую сторону направлен хвост.

Реализуйте функцию, которая должна вернуть строку `"RIGHT"`, `"LEFT"` или `"SYMMETRIC"` в зависимости от направления хвоста.

### Примеры

```text
[1, 2, 3]       -> SYMMETRIC
[1, 1, 1, 10]   -> RIGHT
[-10, 1, 1, 1]  -> LEFT
```

Для пустого массива верните `"UNDEFINED"`.

In [11]:
def skewness_direction(values):
    """Определяет направление асимметрии по третьему центральному моменту."""
    # TODO
    if not values:
        return "UNDEFINED"
    s = 0
    s1 = 0
    for i in values:
        s += i
    av = s / len(values)
    for i in values:
        s1 += (i - av) * (i - av) * (i - av)
    u = s1 / len(values)
    if u > 0:
        return "RIGHT"
    elif u == 0:
        return "SYMMETRIC"
    else:
        return"LEFT"

In [12]:
%run -i ./tests/check_0c_skewness_direction.py

OK: задача 0C


## Разминка. Префиксы трёх степеней

Для массива `a` постройте три массива:

$$
P_1[i]=\sum_{j=1}^{i}a_j\qquad - обычный массив префиксных сумм\\
P_2[i]=\sum_{j=1}^{i}a_j^2\qquad - массив, хранящий сумму квадратов на префиксе \\ 
P_3[i]=\sum_{j=1}^{i}a_j^3 \qquad - массив, хранязий сумму кубов на префиксе
$$

У каждого префикса первый элемент равен `0`.

**Сигнатура:** `build_prefixes(values)`

Верните `(p1, p2, p3)`.

In [21]:
def build_prefixes(values):
    """Префиксы сумм x, x**2 и x**3."""
    # TODO
    p1 = [0]
    p2 = [0]
    p3 = [0]
    for i in values:
        p1.append(i + p1[-1])
        p2.append(i * i + p2[-1])
        p3.append(i * i * i + p3[-1])
    return p1, p2, p3

In [22]:
%run -i ./tests/check_prefixes.py

OK: префиксы


# Задача A. Средняя температура

За `n` часов получили последовательность температур. Для каждого запроса `[l, r]`
найдите среднее значение на этом отрезке.

### Входные данные

В функцию передаются:

- `n` — число элементов;
- `values` — массив из `n` целых чисел;
- `q` — число запросов;
- `queries` — список пар `[l, r]`.

### Выходные данные

Верните строку: для каждого запроса среднее значение с **6 знаками после точки**,
по одному ответу на строку.

### Ограничения

- `1 <= n, q <= 100000`
- `-10000 <= values[i] <= 10000`
- `1 <= l <= r <= n`

Ожидаемая сложность: `O(n + q)`.

### Пример

```text
values = [2, 4, 6, 8, 10]
queries = [[1, 3], [2, 5]]

4.000000
7.000000
```

In [46]:
def solve_mean_queries(n, values, q, queries):
    # TODO
    s = [0]
    res =  ""
    for i in values:
        s.append(i + s[-1])
    for i in range(len(queries)):
        q = len(queries)
        r = queries[i][1]
        l = queries[i][0] - 1
        s1 = s[r] - s[l]
        av = s1 / (r - l)
        res += (f"{float(round(av, 6))}\n")
    return res

    

In [47]:
%run -i ./tests/check_a_mean.py

OK: задача A


# Задача B. Насколько скачет сигнал?

Для каждого запроса `[l, r]` найдите дисперсию элементов отрезка.

Откуда появилась формула ниже? Используйте для решения задачи.

$$
D(x)=E(x^2)-E(x)^2.
$$

$D(x)$ - дисперсия на отрезке, $E(x)$ - среднее арифметическое числе на отрезке, $E(x^2)$ - средняя сумма квадратов чисел на отрезке, $E(x^3)$ - средняя сумма кубов чисел на отрезке.

### Выходные данные

Для каждого запроса выведите дисперсию с **6 знаками после точки**.

### Ограничения

- `1 <= n, q <= 100000`
- `-10000 <= values[i] <= 10000`
- `1 <= l <= r <= n`

Ожидаемая сложность: `O(n + q)`.

### Пример

```text
values = [1, 2, 3, 4]
queries = [[1, 4], [2, 3]]

1.250000
0.250000
```

Из-за вычислений с `float` очень маленькое отрицательное число вроде `-1e-15`
можно считать нулём.

In [48]:
def solve_variance_queries(n, values, q, queries):
    # TODO
    s = [0]
    s1 = [0]
    res = ""
    for i in values:
        s.append(i + s[-1])
        s1.append(i * i + s1[-1])
    for i in range(len(queries)):
        r = queries[i][1]
        l = queries[i][0] - 1
        var1 = (s1[r] - s1[l]) / (r - l) - ((s[r] - s[l]) / (r - l)) * ((s[r] - s[l]) / (r - l))
        res += f"{float(round(var1, 6))}\n"
    return res

In [49]:
%run -i ./tests/check_b_variance.py

OK: задача B


# Задача C. Какой участок стабильнее?

В каждом запросе даны два отрезка:

`[l1, r1]` и `[l2, r2]`.

Стабильнее тот участок, у которого **меньше дисперсия**.

### Входные данные

Каждый запрос имеет вид:

```text
[l1, r1, l2, r2]
```

### Выходные данные

- `FIRST` — первый отрезок стабильнее;
- `SECOND` — второй отрезок стабильнее;
- `EQUAL` — дисперсии равны с точностью `1e-9`.

### Ограничения

- `1 <= n, q <= 100000`
- все границы корректны.

Ожидаемая сложность: `O(n + q)`.

### Пример

```text
values = [5, 5, 5, 1, 3, 5]
queries = [[1, 3, 4, 6]]

FIRST
```

In [50]:
def solve_stability_queries(n, values, q, queries):
    # TODO
    s = [0]
    s1 = [0]
    res = ""
    for i in values:
        s.append(i + s[-1])
        s1.append(i * i + s1[-1])
    for i in range(len(queries)):
        r = queries[i][1]
        l = queries[i][0] - 1
        r1 = queries[i][3]
        l1 = queries[i][2] - 1
        var1 = (s1[r] - s1[l]) / (r - l) - ((s[r] - s[l]) / (r - l)) * ((s[r] - s[l]) / (r - l))
        var2 = (s1[r1] - s1[l1]) / (r1 - l1) - ((s[r1] - s[l1]) / (r1 - l1)) * ((s[r1] - s[l1]) / (r1 - l1))
        if var1 < var2:
            res += "FIRST\n"
        elif var1 == var2:
            res += "EQUAL\n"
        else:
            res += "SECOND\n"
    return res

In [51]:
%run -i ./tests/check_c_stability.py

OK: задача C


# Задача D. Тревога датчика

Каждый запрос имеет вид `[l, r, t]`.

Нужно вывести `YES`, если стандартное отклонение на отрезке строго больше `t`,
иначе `NO`.

$$
\sigma=\sqrt{D(X)}.
$$

Можно заметить, что при `t >= 0`:

$$
\sigma>t \iff D(X)>t^2.
$$

### Ограничения

- `1 <= n, q <= 100000`
- `0 <= t <= 10000`

Ожидаемая сложность: `O(n + q)`.

### Пример

```text
values = [1, 1, 1, 5]
queries = [[1, 3, 0], [1, 4, 1]]

NO
YES
```

In [54]:
def solve_alarm_queries(n, values, q, queries):
    # TODO
    s = [0]
    s1 = [0]
    res = ""
    for i in values:
        s.append(i + s[-1])
        s1.append(i * i + s1[-1])
    for i in range(len(queries)):
        r = queries[i][1]
        l = queries[i][0] - 1
        t = queries[i][2]
        var1 = (s1[r] - s1[l]) / (r - l) - ((s[r] - s[l]) / (r - l)) * ((s[r] - s[l]) / (r - l))
        if var1 > t * t:
            res += "YES\n"
        else:
            res += "NO\n"
    return res

In [55]:
%run -i ./tests/check_d_alarm.py

OK: задача D


# Задача E***. Асимметричный участок

Для каждого запроса `[l, r]` найдите коэффициент асимметрии:

$$
\gamma_1=\frac{\mu_3}{\sigma^3}.
$$

Попробуйте самостоятельно преобразовать формулу выше через ранее посчитанные префиксные суммы.

Если дисперсия равна нулю, асимметрия не определена.

### Выходные данные

Для каждого запроса:

- число с **9 знаками после точки**;
- `UNDEFINED`, если дисперсия равна нулю.

### Ограничения

- `1 <= n, q <= 100000`
- `-1000 <= values[i] <= 1000`

Допустимая погрешность числового ответа: `1e-6`.

Ожидаемая сложность: `O(n + q)`.

### Пример

```text
values = [1, 2, 3, 4]
queries = [[1, 4], [2, 2]]

0.000000000
UNDEFINED
```

In [70]:
import math


def solve_skewness_queries(n, values, q, queries):
    # TODO
    s = [0]
    s1 = [0]
    s2 = [0]
    res = ""
    for i in values:
        s.append(i + s[-1])
        s1.append(i * i + s1[-1])
        s2.append(i * i * i + s2[-1])
    for i in range(len(queries)):
        r = queries[i][1]
        l = queries[i][0] - 1
        u3 = (s2[r] - s2[l]) / (r - l) - 3 * ((s[r] - s[l]) / (r - l)) * ((s1[r] - s1[l]) / (r - l)) + 2 * (((s[r] - s[l]) / (r - l)) ** 3)
        var1 = (s1[r] - s1[l]) / (r - l) - ((s[r] - s[l]) / (r - l)) * ((s[r] - s[l]) / (r - l))
        if var1 < 0.000000001:
            res += "UNDEFINED\n"
        else:
            var3 = var1 * math.sqrt(var1)
            res += f"{float(round(u3 / var3, 9))}\n"

    return res

In [71]:
%run -i ./tests/check_e_skewness.py

OK: задача E


## Финальная проверка

Перед сдачей перезапустите ядро и выполните весь ноутбук сверху вниз.
Последняя ячейка повторно запускает все тесты.

In [72]:
%run -i ./tests/check_all.py

--- check_0a_mean_filter.py ---
OK: задача 0A
--- check_0b_std_filter.py ---
OK: задача 0B
--- check_0c_skewness_direction.py ---
OK: задача 0C
--- check_prefixes.py ---
OK: префиксы
--- check_a_mean.py ---
OK: задача A
--- check_b_variance.py ---
OK: задача B
--- check_c_stability.py ---
OK: задача C
--- check_d_alarm.py ---
OK: задача D
--- check_e_skewness.py ---
OK: задача E

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
